# Backpropagation Derived

Part of **ML Math Applied**.

## 1. Intuition First

Backpropagation is the multivariable chain rule executed in reverse topological order.

![Backpropagation chain rule](../assets/diagrams/backprop_chain_rule.svg)

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=5, suppress=True)
torch.set_printoptions(precision=5)

## 2. Derive the Gradients for a Two-Layer MLP

Consider

$$
z_1 = XW_1 + b_1,
\qquad
h = \mathrm{ReLU}(z_1),
\qquad
\hat{y} = hW_2 + b_2,
$$

with mean-squared loss

$$
L = \frac{1}{2m} \|\hat{y} - y\|_2^2.
$$

Then

$$
\frac{\partial L}{\partial \hat{y}} = \frac{1}{m} (\hat{y} - y),
\quad
\frac{\partial L}{\partial W_2} = h^\top \frac{\partial L}{\partial \hat{y}},
\quad
\frac{\partial L}{\partial h} = \frac{\partial L}{\partial \hat{y}} W_2^\top.
$$

ReLU contributes a diagonal Jacobian:

$$
\frac{\partial L}{\partial z_1} = \frac{\partial L}{\partial h} \odot \mathbf{1}[z_1 > 0],
$$

and the first layer gradients become

$$
\frac{\partial L}{\partial W_1} = X^\top \frac{\partial L}{\partial z_1},
\qquad
\frac{\partial L}{\partial b_1} = \sum_i \frac{\partial L}{\partial z_1^{(i)}}.
$$

In [ ]:
X = np.array([[1.0, -1.0], [0.5, 2.0], [-1.5, 1.0]])
y = np.array([[0.6], [-0.2], [0.8]])

W1 = np.array([[0.4, -0.3, 0.2], [0.1, 0.5, -0.4]])
b1 = np.array([0.05, -0.1, 0.2])
W2 = np.array([[0.7], [-0.2], [0.5]])
b2 = np.array([0.1])

z1 = X @ W1 + b1
h = np.maximum(z1, 0.0)
y_hat = h @ W2 + b2
residual = y_hat - y
loss = 0.5 * np.mean(residual**2)

dL_dyhat = residual / len(X)
dL_dW2 = h.T @ dL_dyhat
dL_db2 = dL_dyhat.sum(axis=0)
dL_dh = dL_dyhat @ W2.T
dL_dz1 = dL_dh * (z1 > 0.0)
dL_dW1 = X.T @ dL_dz1
dL_db1 = dL_dz1.sum(axis=0)

print("loss =", loss)
print("dL/dW1 =\n", dL_dW1)
print("dL/dW2 =\n", dL_dW2)

## 3. PyTorch Verification

PyTorch should produce the same gradients when we rebuild the exact graph.

In [ ]:
X_t = torch.tensor(X, dtype=torch.float64)
y_t = torch.tensor(y, dtype=torch.float64)
W1_t = torch.tensor(W1, dtype=torch.float64, requires_grad=True)
b1_t = torch.tensor(b1, dtype=torch.float64, requires_grad=True)
W2_t = torch.tensor(W2, dtype=torch.float64, requires_grad=True)
b2_t = torch.tensor(b2, dtype=torch.float64, requires_grad=True)

y_hat_t = torch.relu(X_t @ W1_t + b1_t) @ W2_t + b2_t
loss_t = 0.5 * torch.mean((y_hat_t - y_t) ** 2)
loss_t.backward()

print(torch.allclose(torch.tensor(dL_dW1), W1_t.grad, atol=1e-8))
print(torch.allclose(torch.tensor(dL_dW2), W2_t.grad, atol=1e-8))
print(torch.allclose(torch.tensor(dL_db1), b1_t.grad, atol=1e-8))
print(torch.allclose(torch.tensor(dL_db2), b2_t.grad, atol=1e-8))

## 4. Custom Figure

The arrow labels show how gradient norms change as the chain rule propagates error backward.

In [ ]:
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.axis("off")

nodes = [((0.08, 0.5), "X"), ((0.30, 0.5), "z1"), ((0.50, 0.5), "h"), ((0.70, 0.5), "y_hat"), ((0.90, 0.5), "L")]
for (x, y0), label in nodes:
    box = FancyBboxPatch((x - 0.055, y0 - 0.11), 0.11, 0.22, boxstyle="round,pad=0.02", facecolor="#e0f2fe", edgecolor="#0f766e", linewidth=2)
    ax.add_patch(box)
    ax.text(x, y0, label, ha="center", va="center", fontsize=14)

arrows = [((0.135, 0.5), (0.245, 0.5), "W1"), ((0.355, 0.5), (0.445, 0.5), "ReLU"), ((0.555, 0.5), (0.645, 0.5), "W2"), ((0.755, 0.5), (0.845, 0.5), "MSE")]
for start, end, label in arrows:
    ax.add_patch(FancyArrowPatch(start, end, arrowstyle="-|>", mutation_scale=20, linewidth=2.5, color="#334155"))
    ax.text((start[0] + end[0]) / 2, 0.58, label, ha="center", fontsize=11)

backward = [
    ((0.845, 0.28), (0.755, 0.28), f"||dL/dy_hat|| = {np.linalg.norm(dL_dyhat):.3f}"),
    ((0.645, 0.28), (0.555, 0.28), f"||dL/dW2|| = {np.linalg.norm(dL_dW2):.3f}"),
    ((0.445, 0.28), (0.355, 0.28), f"||dL/dz1|| = {np.linalg.norm(dL_dz1):.3f}"),
    ((0.245, 0.28), (0.135, 0.28), f"||dL/dW1|| = {np.linalg.norm(dL_dW1):.3f}"),
]
for start, end, label in backward:
    ax.add_patch(FancyArrowPatch(start, end, arrowstyle="-|>", mutation_scale=18, linewidth=2.2, color="#dc2626"))
    ax.text((start[0] + end[0]) / 2, 0.18, label, ha="center", fontsize=10, color="#991b1b")

ax.set_title("Backpropagation is the chain rule applied in reverse")
plt.show()

## 5. Case Study: Gradient Flow in Deep Nets

- if many local Jacobians are smaller than 1, gradients vanish
- if many are larger than 1, gradients explode
- residual connections and normalization layers help keep these products well-scaled